<a href="https://colab.research.google.com/github/PRIYAGUNAR/marine-oil-spill-detection/blob/Model-and-evaluation/visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn

class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c,out_c,3,padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c,out_c,3,padding=1),
            nn.ReLU(inplace=True)
        )
    def forward(self,x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.down1 = DoubleConv(1,32)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(32,64)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(64,128)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(128,256)

        self.up3 = nn.ConvTranspose2d(256,128,2,stride=2)
        self.conv3 = DoubleConv(256,128)

        self.up2 = nn.ConvTranspose2d(128,64,2,stride=2)
        self.conv2 = DoubleConv(128,64)

        self.up1 = nn.ConvTranspose2d(64,32,2,stride=2)
        self.conv1 = DoubleConv(64,32)

        self.final = nn.Conv2d(32,1,1)

    def forward(self,x):
        d1 = self.down1(x)
        p1 = self.pool1(d1)

        d2 = self.down2(p1)
        p2 = self.pool2(d2)

        d3 = self.down3(p2)
        p3 = self.pool3(d3)

        bn = self.bottleneck(p3)

        up3 = self.up3(bn)
        c3 = self.conv3(torch.cat([up3,d3],dim=1))

        up2 = self.up2(c3)
        c2 = self.conv2(torch.cat([up2,d2],dim=1))

        up1 = self.up1(c2)
        c1 = self.conv1(torch.cat([up1,d1],dim=1))

        return self.final(c1)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet().to(device)

checkpoint = torch.load("/content/best_model_1 (1).pth", map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

print("Model loaded successfully")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os

img_dir = "split_dataset/test/images"
mask_dir = "split_dataset/test/masks"

images = os.listdir(img_dir)[:5]

for name in images:

    img = Image.open(os.path.join(img_dir,name)).convert("L")
    mask = Image.open(os.path.join(mask_dir,name)).convert("L")

    img = np.array(img)/255.0
    mask = np.array(mask)/255.0

    img_tensor = torch.tensor(img).unsqueeze(0).unsqueeze(0).float().to(device)

    with torch.no_grad():
        output = model(img_tensor)
        pred = torch.sigmoid(output)
        pred = (pred>0.4).float().cpu().numpy()[0][0]

    plt.figure(figsize=(10,3))

    plt.subplot(1,3,1)
    plt.title("Image")
    plt.imshow(img,cmap="gray")

    plt.subplot(1,3,2)
    plt.title("Ground Truth")
    plt.imshow(mask,cmap="gray")

    plt.subplot(1,3,3)
    plt.title("Prediction")
    plt.imshow(pred,cmap="gray")

    plt.show()

In [ ]:
!ls

In [ ]:
!unzip /content/split_dataset-20260311T172056Z-3-001.zip

In [ ]:
!ls

In [ ]:
!ls split_dataset

In [ ]:
!ls split_dataset/test